# 📝 Day 2 Homework: Building Agents with Google ADK

**Scoring Criteria: 10 points**

| Step | Score | Content |
|---------|-------|--------|
| 1. Agent + Custom Tool | 3 | Build an Agent with a tool that actually works |
| 2. RAG Agent | 3 | Connect the Agent to Qdrant to search personalized data |
| 3. Workflow Agent | 2 | Use Sequential/Parallel/Loop to build a pipeline |
| 4. Explain the Results | 2 | Explain in your own words |

### ⚠️ Rules
- The data is generated from your **student ID** → each student gets different results
- **Do not edit** the specified cells
- You must **run every cell** and show outputs before submitting

> **🇹🇭 Thai Text in This Notebook**
>
> Sample data and queries are in Thai because this workshop teaches Thai RAG. English translations are provided as inline comments (`# "translation"`).

## 📦 Install Dependencies

In [ ]:
%%time
!pip install -q google-adk google-genai sentence-transformers qdrant-client langchain-text-splitters

import hashlib, os, json, random
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')

print('✅ Installation complete!')

In [ ]:
%%time
# Configure Gemini API
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ Loaded API Key from Colab Secrets')
except Exception:
    api_key = input('🔑 กรุณาวาง Gemini API Key: ')  # "🔑 Please paste your Gemini API Key: "
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'
print('✅ API Key configured successfully')

## 🎓 Enter Student Information

In [ ]:
# ─── Fill in your information ───
STUDENT_NAME = ''   # เช่น 'สมชาย ใจดี'  # "   # e.g. "
STUDENT_ID   = ''   # เช่น '6512345678'  # "   # e.g. "
PHONE        = ''   # เช่น '081-234-5678'  # "   # e.g. "
LINE_ID      = ''   # เช่น 'somchai.j'  # "   # e.g. "

# ─── Validation (do not edit) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'  # "❌ Please enter your student ID!"
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'  # "❌ Please enter your full name!"

print(f'✅ Full name: {STUDENT_NAME}')
print(f'✅ Student ID: {STUDENT_ID}')
print(f'📱 Phone: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 Generate Personalized Dataset (Do not edit)

> The data is generated from your student ID → everyone gets different data

In [ ]:
# ===== Do not edit this cell =====
random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_topics = [
    {'topic': 'healthcare', 'text': 'โรงพยาบาลศิริราช นำ AI มาช่วยวิเคราะห์ภาพ X-ray ทรวงอก ลดเวลาวินิจฉัยจาก 30 นาทีเหลือ 5 นาที ระบบ Deep Learning ตรวจจับความผิดปกติได้แม่นยำ 95% ช่วยแพทย์คัดกรองผู้ป่วยวัณโรคได้เร็วขึ้น'},  # "topic"
    {'topic': 'banking', 'text': 'ธนาคารกสิกรไทย พัฒนา KADE AI Assistant ใช้ RAG ดึงข้อมูลผลิตภัณฑ์ทางการเงินมาตอบลูกค้า ลดเวลาตอบคำถามจาก 5 นาทีเหลือ 30 วินาที รองรับภาษาไทยและอังกฤษ'},  # "topic"
    {'topic': 'education', 'text': 'จุฬาลงกรณ์มหาวิทยาลัย สร้างระบบ RAG ถาม-ตอบจากตำราเรียน แปลง PDF กว่า 500 เล่มเป็น vector embeddings ใช้ multilingual model รองรับภาษาไทย'},  # "topic"
    {'topic': 'agriculture', 'text': 'กรมวิชาการเกษตร ใช้ AI วิเคราะห์ภาพถ่ายโดรน ตรวจจับโรคพืชในนาข้าว ความแม่นยำ 92% ช่วยลดความเสียหายจากโรคพืชได้ 40%'},  # "topic"
    {'topic': 'logistics', 'text': 'Kerry Express ใช้ AI วางแผนเส้นทางขนส่งพัสดุ ลดระยะทางขนส่ง 15% ประหยัดน้ำมัน 200 ล้านบาทต่อปี ใช้ Machine Learning ทำนายปริมาณพัสดุล่วงหน้า'},  # "topic"
    {'topic': 'retail', 'text': 'เซ็นทรัล รีเทล ใช้ AI Recommendation System แนะนำสินค้าให้ลูกค้า ยอดขายเพิ่ม 25% วิเคราะห์พฤติกรรมการซื้อจากข้อมูลกว่า 10 ล้านรายการ'},  # "topic"
    {'topic': 'energy', 'text': 'การไฟฟ้าฝ่ายผลิต (EGAT) ใช้ AI พยากรณ์ความต้องการไฟฟ้า ลดการสูญเสียพลังงาน 8% ใช้ Deep Learning วิเคราะห์ข้อมูลจาก Smart Meter ทั่วประเทศ'},  # "topic"
    {'topic': 'tourism', 'text': 'ททท. พัฒนา AI Chatbot ให้ข้อมูลท่องเที่ยวไทย รองรับ 5 ภาษา ให้บริการนักท่องเที่ยวกว่า 1 ล้านคนต่อเดือน ลดภาระ call center ได้ 60%'},  # "topic"
    {'topic': 'insurance', 'text': 'เมืองไทยประกันชีวิต ใช้ AI ประเมินความเสี่ยงสุขภาพ วิเคราะห์ข้อมูลการเคลม ลดเวลาพิจารณาจาก 7 วันเหลือ 1 วัน ตรวจจับ fraud ได้ 90%'},  # "topic"
    {'topic': 'manufacturing', 'text': 'SCG ใช้ AI ตรวจสอบคุณภาพสินค้า Computer Vision ตรวจจับข้อบกพร่อง ลดของเสีย 30% Predictive Maintenance ทำนายการเสียของเครื่องจักรล่วงหน้า 2 สัปดาห์'},  # "topic"
]

random.shuffle(all_topics)
my_topics = all_topics[:6]  # เลือก 6 จาก 10 หัวข้อ

# Prepare data for Qdrant
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)

my_chunks = []
my_sources = []
for item in my_topics:
    chunks = splitter.split_text(item['text'])
    for c in chunks:
        my_chunks.append(c)
        my_sources.append(item['topic'])

# Embed + Index
passages = [f'passage: {c}' for c in my_chunks]
embeddings = embed_model.encode(passages)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
collection_name = f'hw2_{STUDENT_ID}'
qdrant.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

points = [
    models.PointStruct(id=i, vector=embeddings[i].tolist(),
                       payload={'text': my_chunks[i], 'source': my_sources[i]})
    for i in range(len(my_chunks))
]
qdrant.upsert(collection_name=collection_name, points=points)

# Create personalized query
query_pool = [
    'AI ช่วยลดต้นทุนในธุรกิจอย่างไร',  # "How does AI help reduce business costs?"
    'ระบบอัตโนมัติในองค์กรไทย',  # "Automation systems in Thai organizations"
    'การวิเคราะห์ข้อมูลด้วย Machine Learning',  # "Data analysis with Machine Learning"
    'Chatbot สำหรับบริการลูกค้า',  # "Chatbots for customer service"
    'Computer Vision ในอุตสาหกรรมไทย',  # "Computer Vision in Thai industry"
    'การพยากรณ์ด้วย Deep Learning',  # "Forecasting with Deep Learning"
]
random.shuffle(query_pool)
MY_QUERY = query_pool[0]

# Select tool type
tool_types = [
    {'name': 'คำนวณ ROI ของการลงทุน AI', 'func': 'calculate_ai_roi', 'params': 'investment_baht: float, savings_per_year: float, years: int'},  # "name"
    {'name': 'ประเมินความพร้อม AI', 'func': 'assess_ai_readiness', 'params': 'num_employees: int, has_data: bool, budget_million: float'},  # "name"
    {'name': 'คำนวณ Accuracy Score', 'func': 'calculate_accuracy', 'params': 'true_positives: int, false_positives: int, false_negatives: int, true_negatives: int'},  # "name"
    {'name': 'แนะนำ AI Model', 'func': 'recommend_model', 'params': 'task_type: str, data_size: str, budget: str'},  # "name"
]
random.shuffle(tool_types)
MY_TOOL = tool_types[0]

# Workflow type
workflow_types = ['SequentialAgent', 'ParallelAgent', 'LoopAgent']
random.shuffle(workflow_types)
MY_WORKFLOW = workflow_types[0]

print(f'✅ Personalized data is ready!')
print(f'   📊 Topics: {[t["topic"] for t in my_topics]}')
print(f'   📦 Chunks: {len(my_chunks)} | Collection: {collection_name}')
print(f'   🔍 Required query: "{MY_QUERY}"')
print(f'   🔧 Tool to build: {MY_TOOL["name"]}')
print(f'   🔄 Required workflow: {MY_WORKFLOW}')

## 🔧 Helper Functions (Do not edit)

In [ ]:
# ===== Do not edit this cell =====
async def chat_with_agent(agent, message, user_id='user_1', session_id=None):
    """Send a message to the Agent and receive the res..."""
    if session_id is None:
        session_id = f'session_{id(agent)}'
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session = await runner.session_service.create_session(
        app_name=agent.name, user_id=user_id, session_id=session_id)
    content = types.Content(role='user', parts=[types.Part.from_text(text=message)])

    final_response = ''
    tool_calls = []
    async for event in runner.run_async(
        user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text: final_response = part.text
                if part.function_call:
                    tool_calls.append(part.function_call.name)

    if tool_calls:
        print(f'  🔧 Tools: {tool_calls}')
    return final_response, session.id

print('✅ Chat function is ready')


---
## 🎯 Step 1: Create an Agent + Custom Tool (3 points)

Create an Agent with **2 tools**:
1. `search_knowledge` — search for information from Qdrant (code provided)
2. **Custom Tool** — as assigned by the system (see `MY_TOOL`)

### Criteria
- 1 point: create a custom tool that actually works (with docstring + type hints + returns dict)
- 1 point: create an Agent that can use both tools
- 1 point: test and show that the Agent selects the correct tool

In [ ]:
# RAG Tool (provided — do not edit)
def search_knowledge(query: str) -> dict:
    """Search for information about AI in Thailand from the knowledge base
    
    Args:
        query: คำถามที่ต้องการค้นหา
    """
    q_vec = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points(
        collection_name=collection_name, query=q_vec, limit=3).points
    return {
        'status': 'success',
        'results': [{'text': r.payload['text'], 'source': r.payload['source'],
                     'score': round(r.score, 4)} for r in results]
    }

print('✅ RAG Tool is ready')
print(f'\n📌 Tool you must build: {MY_TOOL["name"]}')
print(f'   Function: {MY_TOOL["func"]}({MY_TOOL["params"]})')
print(f'   Must return dict')

In [ ]:
# Step 1: Write your code here

# 💡 Hint:
#   1. Create the function specified by MY_TOOL
#   2. It must have a docstring (LLM reads it!)
#   3. It must have type hints
#   4. It must return dict

# def MY_TOOL['func'](...) -> dict:
#     """Explain..."""
#     ...
#     return {...}



# Create an Agent with 2 tools
# my_agent = Agent(
#     name='my_agent',
#     model='gemini-2.5-flash',
#     tools=[search_knowledge, my_custom_tool]
# )



# Test Tool Selection (ask at least 3 questions)
# 1. A question that must use the RAG tool
# 2. A question that must use the custom tool
# 3. A general question (no tool needed)
# await chat_with_agent(my_agent, '...')

---
## 🎯 Step 2: RAG Agent — Search Personalized Information (3 points)

Use the Agent to search information from Qdrant with the **query assigned by the system**

### Criteria
- 1 point: Agent can search + show Top-3 scores
- 1 point: The answer references information from the knowledge base
- 1 point: Explain in your own words why these chunks were retrieved (3-5 sentences)

In [ ]:
# Step 2: Write your code here

print(f'🔍 Required query: "{MY_QUERY}"')

# 💡 Hint:
#   1. Create an Agent with instructions to search using search_knowledge
#   2. Ask with MY_QUERY
#   3. Show Top-3 scores
#   4. Write your own explanation (do not use AI)

# rag_agent = Agent(
#     name='rag_agent',
#     model='gemini-2.5-flash',
#     instruction='...',
#     tools=[search_knowledge]
# )
# response, _ = await chat_with_agent(rag_agent, MY_QUERY)
# print(response)

In [ ]:
# Step 2 (continued): Explain in your own words
# Write 3-5 sentences explaining:
# 1. Why were these chunks retrieved?
# 2. How is the retrieved information related to the query?
# 3. If you wanted to improve retrieval quality, what should you do?

MY_EXPLANATION_STEP2 = '''
(เขียนคำอธิบายที่นี่)
'''
print(MY_EXPLANATION_STEP2)

---
## 🎯 Step 3: Workflow Agent (2 points)

Create a Workflow based on the **pattern assigned by the system** in `MY_WORKFLOW`

### Criteria
- 1 point: Create the Workflow Agent with the correct pattern
- 1 point: Test it successfully and show the result

In [ ]:
# Step 3: Write your code here

print(f'🔄 Required workflow: {MY_WORKFLOW}')

# 💡 Pattern hints:
#
# If SequentialAgent:
#   from google.adk.agents import SequentialAgent
#   pipeline = SequentialAgent(sub_agents=[agent1, agent2, agent3])
#   → Search → Summarize → Translate (step by step)
#
# If ParallelAgent:
#   from google.adk.agents import ParallelAgent
#   parallel = ParallelAgent(sub_agents=[search1, search2, search3])
#   → Search multiple topics at the same time
#
# If LoopAgent:
#   from google.adk.agents import LoopAgent
#   loop = LoopAgent(sub_agents=[writer, critic], max_iterations=3)
#   → Write ↔ Review until it passes



# Test
# response, _ = await chat_with_agent(my_workflow, MY_QUERY)
# print(response)

---
## 🎯 Step 4: Explain the Results (2 points)

### Criteria
- 1 point: Explain the selected Workflow — why is it suitable for the task? What are its pros/cons?
- 1 point: Compare it with the other 2 patterns — how would the result differ if you used another pattern?

In [ ]:
# Step 4: Write your explanation

MY_EXPLANATION_STEP4 = f'''
Pattern ที่ใช้: {MY_WORKFLOW}

1. ทำไม {MY_WORKFLOW} เหมาะกับงานนี้:
   (เขียนที่นี่)

2. ข้อดีของ {MY_WORKFLOW}:
   (เขียนที่นี่)

3. ข้อเสียของ {MY_WORKFLOW}:
   (เขียนที่นี่)

4. ถ้าใช้ pattern อื่นจะต่างอย่างไร:
   (เขียนที่นี่)
'''
print(MY_EXPLANATION_STEP4)

## ✅ Check Your Answers

In [ ]:
# ===== Do not edit this cell =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_day2_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 Full name: {STUDENT_NAME}')
print(f'🎓 Student ID: {STUDENT_ID}')
print(f'📱 Phone: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 Submit before: _________________ (set by instructor)')
print('=' * 50)
print()
print(f'📌 Personalized data:')
print(f'   Topics: {[t["topic"] for t in my_topics]}')
print(f'   Query: {MY_QUERY}')
print(f'   Tool: {MY_TOOL["name"]}')
print(f'   Workflow: {MY_WORKFLOW}')
print()
print('📋 Checklist before submission:')
print('  [ ] Completed all personal information')
print('  [ ] Step 1: Created Agent + Custom Tool + tested with 3 questions')
print('  [ ] Step 2: RAG Agent + Top-3 scores + 3-5 sentence explanation')
print('  [ ] Step 3: Workflow Agent according to the assigned pattern')
print('  [ ] Step 4: Explained results + comparison')
print('  [ ] Every cell has been run and shows output')